[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S13/S13_01_visualizacion_resultados.ipynb)

# S13 · 01 — Contar lo que hace el modelo: visualización de resultados

**Tiempo estimado:** 1 h · **Módulo:** M5 (Visualización de resultados)

**Contexto TurisData.** Ya tenemos un modelo que predice qué reservas se cancelarán. Pero dirección no
quiere oír *"AUC = 0,74"*: quiere saber **si puede fiarse, qué mueve las cancelaciones y qué decisión
tomar el lunes**. En este laboratorio generamos las visualizaciones estándar de un modelo de
clasificación y aprendemos a convertirlas en un mensaje para personas no técnicas.

**Al terminar sabrás:**
1. Leer y dibujar la matriz de confusión, la curva ROC y la curva precisión-exhaustividad (PR).
2. Comparar dos modelos en un mismo gráfico.
3. Explicar qué variables importan con la **importancia por permutación**.
4. Elegir un **umbral de decisión** en función del coste del negocio.
5. Aplicar buenas prácticas de comunicación: un gráfico, un mensaje.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score, recall_score)
from sklearn.inspection import permutation_importance

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid", font_scale=1.0)

## 0. Punto de partida: dos modelos sencillos
Predecimos `cancelada` (1 = la reserva se cancela). Entrenamos una **regresión logística** y un
**bosque aleatorio** con el 75 % de las reservas y guardamos el 25 % para evaluar.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)

categoricas = X.select_dtypes("object").columns.tolist()
numericas = [c for c in X.columns if c not in categoricas]
prep = ColumnTransformer([("num", StandardScaler(), numericas),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])

modelo_lr = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)
modelo_rf = Pipeline([("prep", prep), ("m", RandomForestClassifier(300, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2))]).fit(X_ent, y_ent)

prob_lr = modelo_lr.predict_proba(X_test)[:, 1]     # probabilidad de cancelar
prob_rf = modelo_rf.predict_proba(X_test)[:, 1]
print(f"Reservas de prueba: {len(y_test)} · tasa real de cancelación: {y_test.mean():.1%}")

## 1. Matriz de confusión: ¿en qué acierta y en qué falla?

**Idea.** Cruza lo que **pasó** con lo que el modelo **dijo** (con un umbral, p. ej. "cancela si
probabilidad ≥ 0,5"): aciertos y dos tipos de error.
- **Falso negativo:** el modelo dijo "no cancela" y canceló → habitación vacía que se pudo revender.
- **Falso positivo:** dijo "cancela" y no canceló → molestamos al cliente o sobrevendemos sin motivo.

Para dirección, **normalizar por filas** (porcentajes de cada clase real) se lee mucho mejor que los
números brutos.

### Ejercicio 1
Calcula `cm` con `confusion_matrix(y_test, pred_lr, normalize="true")`, siendo `pred_lr` la predicción de
la logística con umbral 0,5 (`prob_lr >= 0.5`). Guarda en `recall_cancelaciones` la proporción de
cancelaciones reales que el modelo detecta (fila de las cancelaciones, columna "cancela").

In [ ]:
# TODO: Convierte la comparación prob_lr >= 0.5 en enteros; el recall de la clase 1 es un elemento de la matriz normalizada por filas
...

assert cm.shape == (2, 2) and np.allclose(cm.sum(axis=1), 1), "cm debe ser 2x2 y cada fila sumar 1 (normalize='true')"
assert abs(recall_cancelaciones - recall_score(y_test, pred_lr)) < 1e-9, "recall_cancelaciones = fila 1, columna 1 de cm"
print(f"Ejercicio 1 correcto ✔ — el modelo detecta el {recall_cancelaciones:.0%} de las cancelaciones")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=["No cancela", "Cancela"]).plot(ax=ax, cmap="Blues", values_format=".0%", colorbar=False)
ax.set_title("Matriz de confusión (% de cada clase real)")
ax.set_xlabel("Lo que dijo el modelo"); ax.set_ylabel("Lo que ocurrió")
plt.tight_layout(); plt.show()

## 2. Curvas ROC y precisión-exhaustividad (PR)

**Idea.** La matriz depende de **un** umbral. Las curvas resumen **todos** los umbrales posibles:
- **ROC:** eje X = falsas alarmas (de los que no cancelan, ¿a cuántos avisamos?), eje Y = cancelaciones
  detectadas. La diagonal es el azar; cuanto más arriba-izquierda, mejor. **AUC** = área bajo la curva
  (0,5 = azar, 1 = perfecto).
- **PR:** eje X = exhaustividad (*recall*: ¿qué parte de las cancelaciones cazamos?), eje Y =
  precisión (de los avisos, ¿cuántos eran ciertos?). Es más informativa si la clase que buscamos es rara.

### Ejercicio 2
Calcula `auc_lr` y `auc_rf` (`roc_auc_score`) y `ap_lr` y `ap_rf` (`average_precision_score`, el área
de la curva PR) para ambos modelos con las probabilidades `prob_lr` y `prob_rf`. Guarda en
`mejor_modelo` el texto `"Regresión logística"` o `"Bosque aleatorio"` según cuál tenga mayor AUC.

In [ ]:
# TODO: Ambas funciones reciben (valores reales, probabilidades); compara los dos AUC con un if/else
...

assert 0.6 < auc_lr < 0.9 and 0.6 < auc_rf < 0.9, "Los AUC deberían estar entre 0,6 y 0,9; usa y_test y las probabilidades"
assert mejor_modelo in ("Regresión logística", "Bosque aleatorio"), "mejor_modelo debe ser uno de los dos textos indicados"
print(f"Ejercicio 2 correcto ✔ — AUC logística {auc_lr:.3f} · bosque {auc_rf:.3f} → {mejor_modelo}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for nombre, prob, color in (("Regresión logística", prob_lr, "tab:blue"), ("Bosque aleatorio", prob_rf, "tab:orange")):
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax[0].plot(fpr, tpr, color=color, label=f"{nombre} (AUC = {roc_auc_score(y_test, prob):.2f})")
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ax[1].plot(rec, prec, color=color, label=f"{nombre} (AP = {average_precision_score(y_test, prob):.2f})")
ax[0].plot([0, 1], [0, 1], "--", color="gray", label="Azar")
ax[0].set_xlabel("Falsas alarmas (% de no cancelaciones avisadas)"); ax[0].set_ylabel("Cancelaciones detectadas")
ax[0].set_title("Curva ROC"); ax[0].legend(loc="lower right")
ax[1].axhline(y_test.mean(), linestyle="--", color="gray", label="Azar (tasa de cancelación)")
ax[1].set_xlabel("Exhaustividad (cancelaciones detectadas)"); ax[1].set_ylabel("Precisión (avisos que eran ciertos)")
ax[1].set_title("Curva precisión-exhaustividad"); ax[1].legend(loc="upper right"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 3. ¿Qué variables importan? Importancia por permutación

**Idea.** Se **desordena al azar una variable** en los datos de prueba y se mide **cuánto empeora** el
modelo. Si empeora mucho, esa variable era importante. Funciona con cualquier modelo y se calcula sobre
las **variables originales** (no sobre columnas transformadas), que es lo que dirección entiende.
(SHAP, que da explicaciones caso a caso, se verá en el proyecto; la idea es parecida.)

### Ejercicio 3
Calcula la importancia por permutación de `modelo_lr` sobre `X_test`/`y_test` con
`permutation_importance(..., scoring="roc_auc", n_repeats=5, random_state=SEMILLA)`.
Guarda en la `Series` `importancias` la media (`importances_mean`) con el nombre de cada variable como
índice, ordenada de mayor a menor, y en `top3` la lista de las 3 primeras variables.

In [ ]:
# TODO: El resultado tiene .importances_mean; conviértelo en pd.Series con index=X_test.columns y ordénalo
...

assert isinstance(importancias, pd.Series) and len(importancias) == X_test.shape[1], "importancias: una Series con una entrada por variable"
assert importancias.is_monotonic_decreasing, "Ordena de mayor a menor"
assert top3[0] == "antelacion_dias", "La variable más importante debería ser antelacion_dias; revisa el cálculo"
print("Ejercicio 3 correcto ✔ — las 3 más importantes:", top3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
importancias.head(8)[::-1].plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("Caída del AUC al desordenar la variable")
ax.set_title("La antelación de la reserva es lo que más pesa en una cancelación")
plt.tight_layout(); plt.show()

## 4. Elegir el umbral con el coste del negocio

**Idea.** El umbral 0,5 es una convención, no una decisión. Cada error cuesta distinto:
- **Falso negativo** (no lo vimos venir y canceló): habitación vacía que ya no se revende → **50 €** de ingreso perdido.
- **Falso positivo** (avisamos y no cancelaba): tratamos como "en riesgo" a un cliente que sí viene (compensación o descuento para que confirme) → **30 €**.

Recorremos todos los umbrales y elegimos el de **menor coste total**.

### Ejercicio 4
Para cada umbral de `umbrales` (de 0,05 a 0,95), calcula el coste total con las probabilidades
`prob_lr`: `50 × falsos negativos + 30 × falsos positivos`. Guarda los costes en la lista `costes`, el
mejor umbral en `umbral_optimo` y su coste en `coste_minimo`. Guarda también `coste_05`, el coste del
umbral 0,5.

In [ ]:
umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)

def coste_total(prob, umbral, c_fn=50, c_fp=30):
    pred = (prob >= umbral).astype(int)
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    return c_fn * fn + c_fp * fp

# TODO: Una lista por comprensión con la función coste_total ya escrita; np.argmin te da la posición del mínimo
...

assert len(costes) == len(umbrales), "Un coste por umbral"
assert 0.15 <= umbral_optimo <= 0.5, "Como un falso negativo cuesta más que un falso positivo, el umbral óptimo debería ser inferior a 0,5"
assert coste_minimo <= coste_05, "El coste mínimo no puede ser mayor que el del umbral 0,5"
print(f"Ejercicio 4 correcto ✔ — umbral óptimo {umbral_optimo:.2f}: ahorra {coste_05 - coste_minimo:,.0f} € frente al umbral 0,5 en esta muestra")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(umbrales, costes, color="tab:blue")
ax.plot(umbral_optimo, coste_minimo, "o", color="tab:red")
ax.annotate(f"Umbral óptimo {umbral_optimo:.2f}\n{coste_minimo:,.0f} €", (umbral_optimo, coste_minimo),
            xytext=(umbral_optimo + 0.12, coste_minimo + 0.08 * (max(costes) - min(costes))),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("Umbral (avisamos si la probabilidad supera este valor)"); ax.set_ylabel("Coste total en la muestra de prueba (€)")
ax.set_title("Un umbral algo más bajo que 0,5 es el que menos cuesta")
plt.tight_layout(); plt.show()

## 5. Buenas prácticas: de gráfico técnico a gráfico para dirección

Reglas que aplicamos (y que tu informe debe cumplir):
1. **Un gráfico, un mensaje.** El **título es la conclusión** ("Sin depósito se cancela el doble"), no
   la descripción ("Tasa de cancelación por depósito").
2. **Ejes con nombre y unidades**; barras siempre desde 0; sin recortar el eje para exagerar diferencias.
3. **Pocos colores y con sentido**: destaca en un color lo importante y deja el resto en gris.
4. **Nada de 3D ni tartas** con más de 3 trozos; **etiqueta directamente** los valores en vez de obligar a leer la leyenda.
5. **Incertidumbre a la vista:** di con cuántos datos se hizo y qué no se sabe.
6. **Accesible:** colores distinguibles para daltónicos y texto legible al proyectar.

**Ejemplo:** el mismo dato, mal y bien contado.

In [ ]:
tasa_tarifa = reservas.groupby("tarifa")["cancelada"].mean().mul(100)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
# MAL: eje recortado, título descriptivo, sin etiquetas
ax[0].bar(tasa_tarifa.index, tasa_tarifa.values, color=["red", "green"])
ax[0].set_ylim(35, 53); ax[0].set_title("cancelada por tarifa")
ax[0].set_xlabel("")
# BIEN: eje desde 0, mensaje en el título, valores etiquetados, un color de énfasis
barras = ax[1].bar(["Flexible", "No reembolsable"], tasa_tarifa[["flexible", "no_reembolsable"]].values, color=["tab:red", "lightgray"])
ax[1].bar_label(barras, fmt="%.0f %%", padding=3)
ax[1].set_ylim(0, 65); ax[1].set_ylabel("Reservas canceladas (%)")
ax[1].set_title("La tarifa flexible se cancela 15 puntos más")
sns.despine(); plt.tight_layout(); plt.show()

### Ejercicio 5
Crea para dirección un gráfico de barras de la **tasa de cancelación (%) por tipo de `deposito`**
(`sin_deposito`, `parcial`, `total`). Cumple las reglas: figura `fig_dir` con un solo eje, barras desde 0,
eje Y con nombre, valores etiquetados y un **título que sea una conclusión** (más de 25 caracteres).
Usa la variable `tasa_deposito` (ya calculada) para las alturas.

In [ ]:
tasa_deposito = reservas.groupby("deposito")["cancelada"].mean().mul(100).reindex(["sin_deposito", "parcial", "total"])
# TODO: plt.subplots, ax.bar con las tres tasas, ax.bar_label para etiquetar, set_ylim(0, ...) y un título que diga qué hay que concluir
...

ejes = fig_dir.axes
assert len(ejes) == 1, "fig_dir debe tener un solo gráfico"
assert len(ejes[0].get_title()) > 25, "El título debe ser una frase con la conclusión (más de 25 caracteres)"
assert ejes[0].get_ylabel() != "", "Pon nombre al eje Y"
assert ejes[0].get_ylim()[0] <= 0, "Las barras deben empezar en 0"
assert len(ejes[0].patches) >= 3, "Dibuja las 3 barras"
plt.tight_layout(); plt.show()
print("Ejercicio 5 correcto ✔")

## Mini-reto integrador: el panel de una página

Junta en **una sola figura de 2 × 2** (`plt.subplots(2, 2, figsize=(12, 8))`) las cuatro piezas que
necesita dirección: (1) matriz de confusión, (2) curva ROC o PR, (3) importancia de variables y (4) tu
gráfico de negocio. Añade un título general (`fig.suptitle`) con **la decisión que recomiendas**.
Es el borrador del reto "Informe visual para dirección": la plantilla completa está en el notebook
`S13_02_plantilla_informe_visual` y en la guía del sprint.

## Preguntas de reflexión
1. Si TurisData tuviera un 5 % de cancelaciones en lugar de un 46 %, ¿qué curva (ROC o PR) te parecería
   más honesta? ¿Por qué el AUC podría "sonar" bien y ser inútil?
2. ¿Qué enseñarías a dirección: AUC, matriz de confusión o el ahorro en euros? ¿Por qué?
3. La antelación es la variable más importante. ¿Significa que *acortar* la antelación **causa** menos
   cancelaciones? Distingue correlación de causalidad.
4. Si el modelo acertara menos con clientes de ciertos países, ¿cómo lo comprobarías con un gráfico?

## Qué has aprendido
- Matriz de confusión, ROC y PR: qué mide cada una y cuándo usarla.
- Importancia por permutación para explicar **qué** pesa en el modelo.
- El umbral se elige con el coste del negocio, no por costumbre.
- Un buen gráfico tiene un mensaje en el título, ejes honestos y pocos colores.